In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/data-article/ARTICLM_CMR_23JANV.xlsx
/kaggle/input/data-article/C2C3REP_BA_23JANV_ALL.xlsx
/kaggle/input/data-article/VTEDET_CMR_23JANV_ALL.xlsx


In [2]:
!pip install fuzzywuzzy gradio openpyxl

INFO: pip is looking at multiple versions of httpcore2 to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of httpcore2 to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
Requested uvicorn>=0.14.0 from https://files.pythonhosted.org/packages/bf/fe/a41994c92897b162c0c83e8ef10bec54ebdefbce3f3725b530d2091492ac/uvicorn-0.14.0-py3-none-any.whl (from gradio) has invalid metadata: .* suffix can only be used with `==` or `!=` operators
    click (>=7.*)
           ~~~~^
Please use pip<24.1 if you need to use this version.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 68.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━

In [3]:
import pandas as pd
import os
import gradio as gr
import cv2
import io
import zipfile
from fuzzywuzzy import fuzz

/usr/local/lib/python3.10/dist-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


**DM_PRODUCTS**

In [4]:
def process_products_file(products_file):
    try:
        products_data = pd.read_excel(products_file)
        

        products_data.columns = products_data.columns.str.upper()


        output_data = pd.DataFrame({
           
            'PRODUCTNUMBER':'',
            'ISPRODUCTKIT': 'NO',
            'PRODUCTDESCRIPTION': products_data['B9BQT1'].replace("  ", ""),
            'PRODUCTNAME': products_data['B9BQT1'].replace("  ", ""),
            'PRODUCTSEARCHNAME': '',
            'PRODUCTSUBTYPE': 'PRODUCT',
            'PRODUCTTYPE':'ITEM',
            'PRODUCTAS400NUMBER': products_data['B9N6N2'],

        })
        output_file = 'PRODUIT.xlsx'
        output_data.to_excel(output_file, index=False)
        return output_file
    except Exception as e:
        raise Exception(f"Erreur lors de la transformation: {str(e)}")

**DM PRODUCT TRANSLATION**

In [5]:
def process_products_translation(products_file):
    try:
        products_data = pd.read_excel(products_file)

        products_data.columns = products_data.columns.str.upper()
        replacements = {
            'F': 'FR',
            'A': 'En-Us',
            'B': 'AR',
            'C': 'zh-hans',
            'D': 'DE',
            'E': 'ES',
            'P': 'pt-PT',
            'R': 'RU',
            'Z': 'FR'
        }
        
        # Apply the replacements to the column
        products_data['B9CACD'] = products_data['B9CACD'].str.strip().replace(replacements, regex=False)


        output_data = pd.DataFrame({
            'PRODUCTNUMBER':'',
            'LANGUAGEID': products_data['B9CACD'],
            'DESCRIPTION': products_data['B9BRT1'],
            'PRODUCTNAME': products_data['B9BPT1'],
            'PRODUCTLAGNUMBER': products_data['B9N6N2']
            

        })
        output_file = 'PRODUIT_TRANSLATION.xlsx'
        output_data.to_excel(output_file, index=False)
        return output_file
    except Exception as e:
        raise Exception(f"Erreur lors de la transformation: {str(e)}")

**DM CATEGORY ASSIGMNENT**

In [6]:
def get_category_name(row):
    if row['CATMAR'] == 1:
        return "SF"
    else:
        return "DF"

In [7]:
import numpy as np
import pandas as pd

def process_products_category_assignment(products_file):
    try:
        products_data = pd.read_excel(products_file, dtype=str)
        products_data.columns = products_data.columns.str.strip().str.upper()

        categorie = products_data['CLASSE(S) ATC'].fillna('').str.strip()

        output_data = pd.DataFrame({
            'PRODUCTNUMBER': '',
            'PRODUCTCATEGORYNAME': categorie,
            'PRODUCTCATEGORYHIERARCHYNAME': np.where(categorie != '', 'Hierarchy ATC', ''),
            'PRODUCTNUMBER1': products_data['B9N6N2'],
        })

        output_file = 'PRODUIT_CATEGORY_ASSIGNMENT.xlsx'
        output_data.to_excel(output_file, index=False)
        return output_file
    except Exception as e:
        raise Exception(f"Erreur lors de la transformation: {e}") from e

**DM_External item descriptions for vendors**


In [8]:
def process_products_external_items_for_vendors(products_file):
    try:
        products_data = pd.read_excel(products_file)

        products_data.columns = products_data.columns.str.upper()


        output_data = pd.DataFrame({
            'PRODUCTNUMBER':'',
            'VENDORACCOUNTNUMBER': products_data['C3L7NB'],
            'VENDORPRODUCTDESCRIPTION':'',
            'VENDORPRODUCTNUMBER': products_data['C3NNN2']
            

        })
        output_file = 'PRODUIT_External_item_descriptions_for_vendors.xlsx'
        output_data.to_excel(output_file, index=False)
        return output_file
    except Exception as e:
        raise Exception(f"Erreur lors de la transformation: {str(e)}")

In [9]:
def process_products_external_items_for_customers(products_file):
    try:
        products_data = pd.read_excel(products_file)

        products_data.columns = products_data.columns.str.upper()


        output_data = pd.DataFrame({
            #'ITEMNUMBER': products_data['NOART'].apply(lambda x: f"PR{str(x).zfill(11 - 2)}"),
            'ITEMNUMBER':'',
            'CUSTOMERACCOUNTNUMBER': products_data['C3L7NB'],
            'CUSTOMERPRODUCTDESCRIPTION':'',
            'CUSTOMERPRODUCTNUMBER': products_data['C3NNN2']
            

        })
        output_file = 'PRODUIT_External_item_descriptions_for_customers.xlsx'
        output_data.to_excel(output_file, index=False)
        return output_file
    except Exception as e:
        raise Exception(f"Erreur lors de la transformation: {str(e)}")

In [10]:
def get_code_rembss(products_data, rembss_file):
    try:
        rembss_data = pd.read_excel(rembss_file, dtype={'EKVIC1': str})
        rembss_data.columns = rembss_data.columns.str.upper()
        rembss_data['EKVIC1'] = rembss_data['EKVIC1'].str.strip()

        mapping = (rembss_data.drop_duplicates(subset='EKVIC1')
                              .set_index('EKVIC1')['EKIEPC'])

        return products_data['B9VIC1'].astype(str).str.strip().map(mapping)

    except Exception as e:
        raise Exception(f"Erreur lors de la transformation: {e}") from e


In [11]:
def get_code_douane(products_data, rembss_file):
    try:
        products_data = pd.read_excel(products_file, dtype={'B9W0CD ': str})
        products_data.columns = products_data.columns.str.upper()

        rembss_data = pd.read_excel(rembss_file, dtype={'J5W0CD': str}, sheet_name = "code_douane")
        rembss_data.columns = rembss_data.columns.str.upper()

        # Normalisation des clés
        products_data['B9W0CD'] = products_data['B9W0CD'].str.strip()
        rembss_data['J5W0CD'] = rembss_data['J5W0CD'].str.strip()

        mapping = (rembss_data.drop_duplicates(subset='J5W0CD')
                              .set_index('J5W0CD')[' J5W1CD'])
        return products_data['B9W0CD'].astype(str).str.strip().map(mapping)

    except Exception as e:
        raise Exception(f"Erreur lors de la transformation: {e}") from e

In [12]:
import numpy as np
import pandas as pd


def _norm_cols(df):
    df.columns = df.columns.str.strip().str.upper()
    return df


def _clean(s):
    """AS400 CHAR fields are space-padded: strip + NaN -> ''."""
    return s.fillna('').astype(str).str.strip()


def _lookup(keys, ref, key_col, val_col):
    ref = ref.copy()
    ref[key_col] = _clean(ref[key_col])
    mapping = ref.drop_duplicates(subset=key_col).set_index(key_col)[val_col]
    return _clean(keys).map(mapping).fillna('')


def _flag(s, mapping):
    return _clean(s).map(mapping).fillna('')


def process_released_products(products_file, rembss_file):
    try:
        products_data = _norm_cols(pd.read_excel(products_file, dtype=str))
        rembss_data = _norm_cols(pd.read_excel(rembss_file, dtype=str, sheet_name="remboursement"))
        code_douane_data = _norm_cols(pd.read_excel(rembss_file, dtype=str, sheet_name="code_douane"))
        transco_pays = _norm_cols(pd.read_excel(rembss_file, dtype=str, sheet_name="PaysOrigine"))
        code_pospro_data = _norm_cols(pd.read_excel(rembss_file,dtype=str, sheet_name="PositionProduit"))

        def get_code_rembss(products_data, rembss_data):
            try:
                return _lookup(products_data['B9VIC1'], rembss_data, 'EKVIC1', 'EKIEPC')
            except Exception as e:
                raise Exception(f"Erreur get_code_rembss: {e}") from e

        def get_code_douane(products_data, code_douane_data):
            try:
                return _lookup(products_data['B9W0CD'], code_douane_data, 'J5W0CD', 'J5W1CD')
            except Exception as e:
                raise Exception(f"Erreur get_code_douane: {e}") from e

        # TODO: ORIGINCOUNTRYREGIONID -> transco_pays is loaded but not mapped yet.
        # Once the columns are known:
        def get_code_iso(products_data, transco_pays):
            try:
                return _lookup(products_data['B9YFC2'], transco_pays, 'CODEVALEURSOURCEAS400', 'CODEVALEURCIBLED365')
            except Exception as e:
                raise Exception(f"Erreur get_code_douane: {e}") from e
        def get_code_provenance(products_data, transco_pays):
            try:
                return _lookup(products_data['B9YGC2'], transco_pays, 'CODEVALEURSOURCEAS400', 'CODEVALEURCIBLED365')
            except Exception as e:
                raise Exception(f"Erreur get_code_douane: {e}") from e
        def get_pospro(products_data, code_pospro_data):
            try:
               return _lookup(products_data['B9BUCD'], code_pospro_data, 'AGBUCD', 'AGANTX')
            except Exception as e:
                raise Exception(f"Erreur get_code_douane: {e}") from e
        def _fmt_date(s):
            """20230627 -> '27/06/2023 00:00' ; vide ou invalide -> ''"""
            d = pd.to_datetime(_clean(s), format='%Y%m%d', errors='coerce')
            return d.dt.strftime('%d/%m/%Y 00:00').fillna('')
      

        amm = _clean(products_data['B9JOS2'])
        hmd = _clean(products_data['B9JVS2'])
        rfitem = _clean(products_data['B9IZS2'])
        marque = _clean(products_data['B9JST2'])
        dangerous = _clean(products_data['B9JJS2'])
        
        

        output_data = pd.DataFrame({
            'HSOSOCIALSECURITYRATE': get_code_rembss(products_data, rembss_data),
            'HSOBRANDEDGENERIC': _flag(products_data['B9JSS2'], {'P': 'PR', 'G': 'G','':'G'}),
            'HSOGALENICFORM': products_data['GALENICForm'],
            'HSOPRESCRIPTION': np.where(_clean(products_data['B9JXS2']) != '', 'YES', 'NO'),
            'HSOPSYCHOQTY': products_data['B9KGQT'],
            'HSOINSPECTEDITEM': '',
            'HSOINSPECTIONCOMPANY': '',
            'HSOINSPECTIONTYPE': '',
            'ITEMNUMBER': '',
            'PRODUCTSEARCHNAME': '',
            'HSOSTORAGETEMP': '',
            'HSOSUFFIX': '',
            'INVENTORYUNITSYMBOL': 'PCS',
            'ORIGINCOUNTRYREGIONID':  get_code_iso(products_data,transco_pays),
            'PRIMARYVENDORACCOUNTNUMBER':_clean(products_data['NUMSUPP']),
            'PRODUCTGROUPID': '',
            'PRODUCTVOLUME': (pd.to_numeric(products_data['B9KTQT'], errors='coerce')
                              * pd.to_numeric(products_data['B9KVQT'], errors='coerce')
                              * pd.to_numeric(products_data['B9KUQT'], errors='coerce')).fillna(0),
            'PURCHASEUNITSYMBOL': '',
            'SALESPRICE': '',
            'SALESPRICEDATE': '',
            'UNITCOST': '',
            'UNITCOSTDATE': '',
            'GROSSDEPTH': products_data['B9KTQT'],
            'HSOAMMSTART': '',
            'HSODOPINGITEM': '',
            'GROSSPRODUCTHEIGHT': products_data['B9KVQT'],
            'GROSSPRODUCTWIDTH': products_data['B9KUQT'],
            'HSOLPPEND': '',
            'HSOLPPRATE': '',
            'HSOLPPRCODE': '',
            'HSOLPPSTART': '',
            'INTRASTATCOMMODITYCODE': '',  
            'NETPRODUCTWEIGHT': products_data['B9KXQT'],
            'HSOOFFICIALGENERIC': _flag(products_data['B9JIS2'], {'O': 'YES', 'N': 'NO','':'NO'}),
            'PRODUCTLIFECYCLESTATEID': get_pospro(products_data,code_pospro_data),
            'HSOPHARMACATEGORY': '',
            'HSOAMM': np.where(amm == 'A', 'YES', np.where(amm != 'A', 'NO', '')),
            'HSOEXIPIENT': _flag(products_data['B9JPS2'], {'O': 'YES', 'N': 'NO','':'NO'}),
            'HSOSERVICETYPE': '',
            'HSOHUMANVETERINARY': _flag(products_data['B9IYS2'], {'H': 'Humain', 'V': 'Vétérinaire'}),
            'HSOMEDICALDEVICE': np.where(hmd == 'M', 'YES', np.where(hmd != 'M', 'NO', '')),
            'HSOTRADEPRICESTART': '',
            'HSONARCOTIC': np.where(hmd == 'S', 'YES', np.where(hmd != 'S', 'NO', '')),
            'HSOVIGILANCE': '',
            'ALTERNATIVEITEMNUMBER': '',
            'ALTERNATIVEPRODUCTUSAGECONDITION': 'RIEN EN STOCK',  
            'APPROVEDVENDORCHECKMETHOD': '',          
            'ARETRANSPORTATIONMANAGEMENTPROCESSESENABLED': 'YES',
            'ARRIVALHANDLINGTIME': '',  
            'BASEPRICEPURCHASE': '',
            'BASESALESPRICESOURCE': '',
            'BATCHMERGEDATECALCULATIONMETHOD': '',
            'BATCHNUMBERGROUPCODE': '',
            'BESTBEFOREPERIODDAYS': '',
            'BUYERGROUPID': '',
            'CARRYINGCOSTABCCODE': '',
            'COSTCALCULATIONGROUPID': '',
            'COSTCHARGESQUANTITY': '',
            'COSTGROUPID': '',
            'CUSTOMSDESCRIPTION': '',
            'DEFAULTDIRECTDELIVERYWAREHOUSE': '',
            'DEFAULTLEDGERDIMENSIONDISPLAYVALUE': '',
            'DEFAULTORDERTYPE': 'PURCH',
            'DEFAULTRECEIVINGQUANTITY': '',
            'EIGHTHPRODUCTFILTERCODE': '',
            'FIFTHPRODUCTFILTERCODE': '',
            'FIRSTPRODUCTFILTERCODE': '',
            'FIRSTPRODUCTFILTERGROUPID': '',
            'FIXEDCOSTCHARGES': '',
            'FIXEDPURCHASEPRICECHARGES': '',
            'FOURTHPRODUCTFILTERCODE': '',
            'FREIGHTALLOCATIONGROUPID': '',
            'INVENTORYCOUNTINGREASONCODEPOLICYNAME': '',
            'INVENTORYRESERVATIONHIERARCHYNAME': '',
            'ISBATCHNUMBERACTIVE': 'YES',
            'ISDELIVEREDDIRECTLY': '',
            'HSOOFFICIALJOURNALDATE': _fmt_date(products_data['B9CHDX']),
            'ISINSTALLMENTELIGIBLE': '',
            'ISINTERCOMPANYPURCHASEUSAGEBLOCKED': 'NO',
            'ISINTERCOMPANYSALESUSAGEBLOCKED': 'NO',
            'ISMANUALDISCOUNTPOSREGISTRATIONPROHIBITED': '',
            'ISRESTRICTEDFORCOUPONS': '',
            'ISSALESPRICEADJUSTMENTALLOWED': '',
            'ISSALESPRICEINCLUDINGCHARGES': '',
            'ISSALESWITHHOLDINGTAXCALCULATED': '',
            'ISSERIALNUMBERACTIVE': '',
            'ISSHIPALONEENABLED': '',
            'ISUNITCOSTAUTOMATICALLYUPDATED': 'YES',
            'ISUNITCOSTINCLUDINGCHARGES': '',
            'ITEMMODELGROUPID': 'STOCK',
            'ITEMOVERUNDERDELIVERYTOLERANCEGROUPID': '',
            'LANDEDCOSTTYPEGROUPID': '',
            'LOWERWARRANTABLEPRICERANGELIMIT': '',
            'MARGINABCCODE': '',
            'MAXIMUMPICKQUANTITY': '',
            'NINTHPRODUCTFILTERCODE': '',
            'PRODUCTCOVERAGEGROUPID': '',
            'PRODUCTDIMENSIONGROUPNAME': '',
            'PRODUCTNUMBER': '',
            'PRODUCTSUBTYPE': 'PRODUCT',
            'PRODUCTTYPE': 'ITEM',
            'PURCHASECHARGEPRODUCTGROUPID': '',
            'PURCHASECHARGESQUANTITY': '',
            'PURCHASEITEMWITHHOLDINGTAXGROUPCODE': '',
            'PURCHASELINEDISCOUNTPRODUCTGROUPCODE': '',
            'PURCHASEMULTILINEDISCOUNTPRODUCTGROUPCODE': '',
            'PURCHASEOVERDELIVERYPERCENTAGE': '',
            'PURCHASEPRICE': '',
            'PURCHASEPRICEDATE': '',
            'PURCHASEPRICEQUANTITY': '0',
            'PURCHASEPRICETOLERANCEGROUPID': '',
            'PURCHASEPRICINGPRECISION': '',
            'PURCHASEREBATEPRODUCTGROUPID': '',
            'PURCHASESALESTAXITEMGROUPCODE': '',
            'PURCHASESUPPLEMENTARYPRODUCTPRODUCTGROUPID': '',
            'PURCHASEUNDERDELIVERYPERCENTAGE': '',
            'REVENUEABCCODE': '',
            'REVRECBUNDLE': '',
            'REVRECDEFAULTREVENUERECOGNITIONSCHEDULE': '',
            'REVRECEXCLUDEFROMCARVEOUT': '',
            'REVRECMEDIANPRICE': '',
            'REVRECMEDIANPRICEMAXIMUMTOLERANCE': '',
            'REVRECMEDIANPRICEMINIMUMTOLERANCE': '',
            'REVRECREVENUERECOGNITIONENABLED': '',
            'REVRECREVENUETYPE': '',
            'SALESCHARGEPRODUCTGROUPID': '',
            'SALESCHARGESQUANTITY': '',
            'SALESITEMWITHHOLDINGTAXGROUPCODE': '',
            'SALESLINEDISCOUNTPRODUCTGROUPCODE': '',
            'SALESMULTILINEDISCOUNTPRODUCTGROUPCODE': '',
            'SALESOVERDELIVERYPERCENTAGE': '',
            'SALESPRICECALCULATIONCHARGESPERCENTAGE': '',
            'SALESPRICECALCULATIONCONTRIBUTIONRATIO': '',
            'SALESPRICECALCULATIONMODEL': '',
            'SALESPRICEQUANTITY': '',
            'SALESPRICINGPRECISION': '',
            'SALESREBATEPRODUCTGROUPID': '',
            'SALESSALESTAXITEMGROUPCODE': '',
            'SALESSUPPLEMENTARYPRODUCTPRODUCTGROUPID': '',
            'SALESUNDERDELIVERYPERCENTAGE': '',
            'SALESUNITSYMBOL': '',
            'SCALEINDICATOR': '',
            'SEARCHNAME': '',
            'SECONDPRODUCTFILTERCODE': _flag(products_data['B9JKS2'], {'O': 'FROID', 'N': 'Ambiant'}),
            'SECONDPRODUCTFILTERGROUPID': '',
            'SELLENDDATE': '',
            'SELLSTARTDATE': '',
            'SERIALNUMBERGROUPCODE': '',
            'SERVICETYPE': '',
            'SEVENTHPRODUCTFILTERCODE': '',
            'SHELFADVICEPERIODDAYS': '',
            'HSOPSYCHOTROPIC': np.where(hmd == 'Q', 'YES', np.where(hmd != 'Q', 'NO', '')),
            'SIXTHPRODUCTFILTERCODE': '',
            'STORAGEDIMENSIONGROUPNAME': 'WMS',
            'TAREPRODUCTWEIGHT': '',
            'TAXRATETYPE': '',
            'TENTHPRODUCTFILTERCODE': '',
            'THIRDPRODUCTFILTERCODE': '',
            'TRACKINGDIMENSIONGROUPNAME': 'BATCH',
            'TRANSFERORDERLANDEDCOSTGROUPID': '',
            'TRANSFERORDEROVERDELIVERYPERCENTAGE': '10',
            'TRANSFERORDERUNDERDELIVERYPERCENTAGE': '10',
            'UNITCONVERSIONSEQUENCEGROUPID': '',
            'UNITCOSTQUANTITY': '',
            'VALUEABCCODE': '',
            'VENDORINVOICELINEMATCHINGPOLICY': '',
            'VOYAGEARRIVALGROUPID': '',
            'WAREHOUSEBATCHDETAILSCAPTUREPOLICY': '',
            'WAREHOUSEFILTERCODE': '',
            'WAREHOUSEFILTERGROUP': '',
            'WAREHOUSEMOBILEDEVICEDESCRIPTIONLINE1': '',
            'WAREHOUSEMOBILEDEVICEDESCRIPTIONLINE2': '',
            'WARRANTABLEPRICERANGEBASETYPE': '',
            'WARRANTYDURATIONTIME': '',
            'WARRANTYDURATIONTIMEUNIT': '',
            'WILLPICKINGWORKBENCHAPPLYBOXINGLOGIC': '',
            'WILLTOTALPURCHASEDISCOUNTCALCULATIONINCLUDEPRODUCT': '',
            'WILLTOTALSALESDISCOUNTCALCULATIONINCLUDEPRODUCT': '',
            'HSOREEFERITEM': np.where(rfitem == '3', 'YES', np.where(rfitem != '3', 'NO', '')),
            'HSOADMINDOSE': '',
            'HSOAI': '',
            'HSOAMMEND': '',
            'HSOBATCHREFUNDABLE': '',
            'HSOSUBJECTTFR': _flag(products_data['B9KCS2'], {'O': 'YES', 'N': 'NO','':'NO'}),
            'HSOVACCINE': _flag(products_data['B9JLS2'], {'O': 'YES', 'N': 'NO'}),
            'HSOBUYER': '',
            'HSOCATALOGITEM': '',
            'HSOCEMARK': '',
            'HSOCOEF': '',
            'HSOCUSTOMSRATE': '',
            'HSODISTRIBUTOR': '',
            'HSODPAV': '',
            'ISHAZARDOUSMATERIAL': _flag(products_data['B9JJS2'], {'O': 'YES', 'N': 'NO','':'NO'}),
            'SHELFLIFEPERIODDAYS': _clean(products_data['B9O1N2']),
            'HSOHYBRIDDRUG': '',
            'HSOICEPACK': _flag(products_data['B9JKS2'], {'O': 'YES', 'N': 'NO','':'NO'}),
            'HSOICEPACKDURATION': '',
            'HSOACTIVEINGREDIENT': '',
            'HSOBLOODDERIVED': np.where(hmd == 'G', 'YES', np.where(hmd != 'G', 'NO', '')),
            'HSOEXCEPTIONALDRUG': _flag(products_data['B9KAS2'], {'O': 'YES', 'N': 'NO','':'NO'}),
            'HSOITEMACTIVITY': '',
            'HSOICPECODE': '',
            'HSOITEMDLVMODE': _flag(products_data['B9B0CD'], {'A': 'AIR', 'M': 'SEA','':'SEA'}),
            'HSOITEMRANGE': '',
            'HSOITEMSOURCE': get_code_provenance(products_data,transco_pays),
            'HSOLABORATORYID': '',
            'HSOLEGACYPRODUCTNUMBER': _clean(products_data['B9N6N2']),
            'HSOLIFECYCLELASTCHANGE': '',
            'HSOLIFECYCLENEXTCHANGE': '',
            'HSOMAD': '',
            'HSOINITHOSPITAL': '',
            'HSOMEDICALSURGICAL': '',
            'HSONSDSCODE': '0',
            'HSOOLDLPPR': '',
            'HSOISCYTOTOXIC': np.where(hmd == 'C', 'YES', np.where(hmd != 'C', 'NO', '')),
            'HSOITEMBRAND': np.where(marque == 'M', 'MARQUE', np.where(marque != 'M', '', '')),
            'HSOPLVTMET': '',
            'HSOPLVTTER': '',
            'HSOMEDDEVCLASS': '',
            'HSOORPHANDRUG': _flag(products_data['B9JMS2'], {'O': 'YES', 'N': 'NO','':'NO'}),
            'HSOPHARMARAWMAT': np.where(hmd == 'Y', 'YES', np.where(hmd != 'Y', 'NO', '')),
            'HSOPRODUCTTYPE': _clean(products_data['TYPEPRODUIT']),
            'HSOPRECURSOR': np.where(hmd == 'Z', 'YES', np.where(hmd != 'Z', 'NO', '')),
            'HSOREGNATURE': '',
            'HSOREPLACINGITEM': '',
            'HSOPRINCEPSGENERIC': _flag(products_data['B9JSS2'], {'P': '1', 'G': '2','':'0'}),
            'HSOTRADEPRICEEND': '',
            'HSOTRANSPORTKO': '',
            'HSOTRANSPORTTEMP': '',
            'HSOABCPHARMACEUTICAL': '',
            'HSOWHOLESALERCATID': '',
        })

        output_file = 'PRODUIT_Released.xlsx'
        output_data.to_excel(output_file, index=False)
        return output_file

    except Exception as e:
        raise Exception(f"Erreur lors de la transformation: {e}") from e

In [13]:
def transform_released_products_create(products_file):
    try:
        products_data = pd.read_excel(products_file)
        products_data.columns = products_data.columns.str.upper()

        output_data = pd.DataFrame({
            'ITEMNUMBER': '',
            'BOMUNITSYMBOL': 'PCS',
            'INVENTORYRESERVATIONHIERARCHYNAME': 'BATCH',
            'INVENTORYUNITSYMBOL': 'PCS',
            'ISPRODUCTKIT': 'NO',
            'ITEMMODELGROUPID': 'STOCK',
            'LOWERWARRANTABLEPRICERANGELIMIT': '',
            'PRODUCTDESCRIPTION': products_data['B9BQT1'].str.strip(),
            'PRODUCTDIMENSIONGROUPNAME': '',
            'PRODUCTGROUPID': '',
            'PRODUCTNAME': products_data['B9BQT1'].str.strip(),
            'PRODUCTNUMBER': '',
            'PRODUCTSEARCHNAME': '',
            'PRODUCTSUBTYPE': 'PRODUCT',
            'PRODUCTTYPE': 'ITEM',
            'PURCHASESALESTAXITEMGROUPCODE': '',
            'PURCHASEUNITSYMBOL': '',
            'RETAILPRODUCTCATEGORYNAME': '',
            'SALESSALESTAXITEMGROUPCODE': '',
            'SALESUNITSYMBOL': '',
            'SEARCHNAME': '',
            'SERVICETYPE': 'NotSpecified',
            'STORAGEDIMENSIONGROUPNAME': 'WMS',
            'TRACKINGDIMENSIONGROUPNAME': 'BATCH',
            'UPPERWARRANTABLEPRICERANGELIMIT': '',
            'VARIANTCONFIGURATIONTECHNOLOGY': '',
            'WARRANTABLEPRICERANGEBASETYPE': '',
            'WARRANTYDURATIONTIME': '',
            'WARRANTYDURATIONTIMEUNIT': ''
        })

        output_file = 'RELEASED_PRODUCT_CREATE.xlsx'
        output_data.to_excel(output_file, index=False)
        return output_file

    except Exception as e:
        raise Exception(f"Erreur lors de la transformation: {str(e)}")

In [14]:
BARCODE_MAPPING = {
    'B9NNN2': 'EAN-13',
    'B9NHNB': 'CIP-7',
    'B9OIN2': 'GTIN',
    'B9UKC1': 'EXTCODE'
}

def process_item_bar_code(products_file):
    try:
        products_data = pd.read_excel(products_file, dtype=str)
        products_data.columns = products_data.columns.str.upper()

        long_df = products_data.reset_index().melt(
            id_vars=['index', 'B9N6N2'],
            value_vars=list(BARCODE_MAPPING.keys()),
            var_name='SOURCE_COL',
            value_name='BARCODE'
        )

        long_df['BARCODE'] = long_df['BARCODE'].fillna('').str.strip()
        long_df = long_df[~long_df['BARCODE'].isin(['', '0'])]
        long_df = long_df.sort_values('index', kind='stable').reset_index(drop=True)

        output_data = pd.DataFrame({
            'ITEMNUMBER': '',
            'PRODUCTQUANTITYUNITSYMBOL': 'PCS',
            'BARCODESETUPID': long_df['SOURCE_COL'].map(BARCODE_MAPPING),
            'ISDEFAULTDISPLAYEDBARCODE': 'NO',
            'ISDEFAULTPRINTEDBARCODE': 'NO',
            'ISDEFAULTSCANNEDBARCODE': 'NO',
            'PRODUCTQUANTITY': 0,
            'BARCODE': long_df['BARCODE'],
            'PRODUCTNUMBER': long_df['B9N6N2'].fillna('').str.strip(),
        })

        output_file = 'item_code_bar.xlsx'
        output_data.to_excel(output_file, index=False)
        return output_file

    except Exception as e:
        raise Exception(f"Erreur lors de la transformation: {e}") from e

In [15]:
def process_approved_vendor_list_by_products(products_file):
    try:
        products_data = pd.read_excel(products_file)

        products_data.columns = products_data.columns.str.upper()


        output_data = pd.DataFrame({
            'PRODUCTNUMBER': products_data['B9N6N2'],
            'ITEMNUMBER':'',
            'APPROVEDVENDORACCOUNTNUMBER': products_data['NUMSUPP'],
            'EFFECTIVEDATE':'',
            'EXPIRATIONDATE': ''
            

        })
        output_file = 'PRODUIT_approved_vendor_list_by_products.xlsx'
        output_data.to_excel(output_file, index=False)
        return output_file
    except Exception as e:
        raise Exception(f"Erreur lors de la transformation: {str(e)}")


In [16]:
def process_default_order_settings(products_file, transco_file):
    try:
        products_data = pd.read_excel(products_file, dtype=str)
        transco_pays = pd.read_excel(transco_file, dtype=str, sheet_name="PaysOrigine")
        products_data.columns = products_data.columns.str.strip().str.upper()
        transco_pays.columns = transco_pays.columns.str.upper()

        try:
            code_iso = _lookup(products_data['B9YFC2'], transco_pays,
                               'CODEVALEURSOURCEAS400', 'CODEVALEURCIBLED365')
        except Exception as e:
            raise Exception(f"Erreur lookup code ISO pays: {e}") from e

        output_data = pd.DataFrame({
            'ITEMNUMBER': '',
            'ISPROCUREMENTPROCESSINGSTOPPED': '',
            'ISPROCUREMENTSITEMANDATORY': 'NO',
            'ISPROCUREMENTUSINGWORKINGDAYS': 'NO',
            'ISPROCUREMENTWAREHOUSEMANDATORY': 'NO',
            'MAXIMUMPROCUREMENTORDERQUANTITY': '',
            'MINIMUMPROCUREMENTORDERQUANTITY': '',
            'PROCUREMENTLEADTIMEDAYS': '',
            'PROCUREMENTQUANTITYMULTIPLES': '',
            'PROCUREMENTSITEID': '',
            'PROCUREMENTWAREHOUSEID': '',
            'ISSALESPROCESSINGSTOPPED': '',
            'STANDARDSALESORDERQUANTITY': '',
            'MULTIPLESALESORDERQUANTITY': '',
            'MINIMUMSALESORDERQUANTITY': '',
            'MAXIMUMSALESORDERQUANTITY': '',
            'PRODUCTNUMBERLAG': products_data['B9N6N2'],
        })

        output_file = 'PRODUIT_default_order_settings.xlsx'
        output_data.to_excel(output_file, index=False)
        return output_file
    except Exception as e:
        raise Exception(f"Erreur lors de la transformation: {e}") from e

**Interface Creation**

In [17]:
with gr.Blocks(css=".footer {visibility: hidden;}") as iface:

        gr.Markdown("Articles Transcoding file tools.")

        with gr.Tab("Products"):
          products_inputs = gr.File(label="Upload Products File")
          products_button = gr.Button("Transcode Products File")
          products_title = "DM PRODUCTS  Transcoding tool"
          products_outputs=gr.File(label="Processed Products File")
          products_button.click(fn=process_products_file, inputs=products_inputs, outputs=products_outputs)
        with gr.Tab("Released Products"):
          products_released_inputs = gr.File(label="Upload Products File")
          products_rembss_inputs = gr.File(label="Upload Transco file for released products")
          released_products_final_inputs = [products_released_inputs,products_rembss_inputs]  
          products_button = gr.Button("Transcode released Products File")
          products_title = "DM PRODUCTS  Transcoding tool"
          products_released_outputs=gr.File(label="Processed released Products File")
          products_button.click(fn=process_released_products, inputs=released_products_final_inputs, outputs=products_released_outputs)
        with gr.Tab("Product bar code"):
          products_item_bar_code_inputs = gr.File(label="Upload Products File")
          products_button = gr.Button("Transcode item bar code Products File")
          products_item_bar_code_title = "DM PRODUCTS  item bar code Transcoding tool"
          products_item_bar_code_outputs=gr.File(label="Processed released Products File")
          products_button.click(fn=process_item_bar_code, inputs=products_item_bar_code_inputs, outputs=products_item_bar_code_outputs)
        with gr.Tab("Products Translation"):
          products_translation_inputs = gr.File(label="Upload Products Translation File")
          products_translation_button = gr.Button("Transcode Products Translation File")
          products_translation_title = "DM PRODUCTS Translation Transcoding tool"
          products_translation_outputs=gr.File(label="Processed Products Translation File")
          products_translation_button.click(fn=process_products_translation, inputs=products_translation_inputs, outputs=products_translation_outputs)
        with gr.Tab("Released Products Create"):
          products_rcreate_inputs = gr.File(label="Upload Products released create File")
          products_rcreate_button = gr.Button("Transcode Products released create File")
          products_rcreate_title = "DM Products released create Transcoding tool"
          products_rcreate_outputs=gr.File(label="Processed Products released create File")
          products_rcreate_button.click(fn=transform_released_products_create, inputs=products_rcreate_inputs, outputs=products_rcreate_outputs)
        with gr.Tab("Products Category Assignment"):
          products_category_inputs = gr.File(label="Upload Products Category Assignment File")
          products_category_button = gr.Button("Transcode Products Category Assignment File")
          products_category_title = "DM Products Category Assignment Transcoding tool"
          products_category_outputs=gr.File(label="Processed Products Category Assignment File")
          products_category_button.click(fn=process_products_category_assignment, inputs=products_category_inputs, outputs=products_category_outputs)
        with gr.Tab("Approved vendor list by products"):
          products_vendor_description_inputs = gr.File(label="Upload Approved vendor list by products File")
          products_vendor_description_button = gr.Button("Transcode Approved vendor list by products File")
          products_vendor_description_title = "DM Approved vendor list by products Transcoding tool"
          products_vendor_description_outputs=gr.File(label="Processed Approved vendor list by products File")
          products_vendor_description_button.click(fn=process_approved_vendor_list_by_products, inputs=products_vendor_description_inputs, outputs=products_vendor_description_outputs)
        with gr.Tab("default order settings"):
          products_default_order_settings_inputs = gr.File(label="Upload default order settings Products File")
          products_rembss_inputs = gr.File(label="Upload Transco file for default order settings products")
          default_order_settings_products_final_inputs = [products_default_order_settings_inputs,products_rembss_inputs]  
          products_button = gr.Button("Transcode default order settings Products File")
          products_title = "DM PRODUCTS default order settings  Transcoding tool"
          products_default_order_settings_outputs=gr.File(label="Processed default order settings Products File")
          products_button.click(fn=process_default_order_settings, inputs=default_order_settings_products_final_inputs, outputs=products_default_order_settings_outputs)  
        with gr.Tab("External item descriptions for customers"):
          products_customers_description_inputs = gr.File(label="Upload External item descriptions for cutomers File")
          products_customers_description_button = gr.Button("Transcode External item descriptions for customers File")
          products_customers_description_title = "DM External item descriptions for customers Transcoding tool"
          products_customers_description_outputs=gr.File(label="Processed External item descriptions for customers File")
          products_customers_description_button.click(fn=process_products_external_items_for_customers, inputs=products_customers_description_inputs, outputs=products_customers_description_outputs) 

        iface.load(lambda: gr.update(css=".footer {visibility: hidden;}"), [], [])

<ipython-input-17-38f5662f9733>:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=".footer {visibility: hidden;}") as iface:


In [ ]:
if __name__ == "__main__":


    iface.queue().launch(debug=True)

* Running on local URL:  http://127.0.0.1:7860
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://3aab8b8e4968797cc2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.10/dist-packages/gradio/blocks.py:2244: UserWarning: A function returned too many output values (needed: 0, returned: 1). Ignoring extra values.
    Output components:
        []
    Output values returned:
        [{'css': '.footer {visibility: hidden;}', '__type__': 'update'}]
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/gradio/blocks.py:2244: UserWarning: A function returned too many output values (needed: 0, returned: 1). Ignoring extra values.
    Output components:
        []
    Output values returned:
        [{'css': '.footer {visibility: hidden;}', '__type__': 'update'}]
  warnings.warn(
